# Step 3 — BAGEL heterogeneous-step GPU correctness

This notebook runs the first GPU gate for M*'s experimental BAGEL heterogeneous-step batching. It compares the unchanged serial path (**B=1**) with a true two-request image-refinement batch (**B=2**) using identical prompts, seeds, geometry, precision, and model weights. The measured requests arrive eight seconds apart so they reach the image loop at different refinement steps.

Success here means: both arms finish, their output images are compared, and the execution trace proves that at least one multi-request `LLM/image_gen` forward contained different per-request loop indices. This is a correctness experiment—not yet a throughput result or a scheduling-policy experiment.

## 1. Mount Drive and define the run

Use a **Colab A100 High-RAM** runtime. The preflight below intentionally stops before downloading the checkpoint if the runtime has less than 75 GiB of GPU memory, 50 GiB of available host memory, or 70 GiB of free local disk.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import json, os, shutil, subprocess, sys

drive.mount('/content/drive')
RUN_ID = 'heterogeneous_parity_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
RUN_COMMIT = 'ff4a90105bf8ffcd25f5554943bf54115c808a4e'
REPO = Path('/content') / f'mstar_{RUN_ID}'
CACHE_DIR = Path('/content/hf_cache_bagel')
RESULT_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL') / RUN_ID
PARITY_DIR = RESULT_DIR / 'heterogeneous_step_parity_b2'
RESULT_DIR.mkdir(parents=True, exist_ok=False)
print({'run_id': RUN_ID, 'repo': str(REPO), 'results': str(RESULT_DIR)})

## 2. Clone the exact candidate and check the runtime

The experiment is pinned to one immutable commit rather than the moving feature branch.

In [ ]:
subprocess.run(['git', 'clone', '--no-checkout', FORK_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', RUN_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == RUN_COMMIT, (head, RUN_COMMIT)
source = {
    'fork_url': FORK_URL,
    'requested_commit': RUN_COMMIT,
    'resolved_commit': head,
    'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True),
}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '70',
], check=True)
print('Hardware preflight passed at commit', head)

## 3. Install and validate the pinned serving environment

This keeps the corrected FlashInfer JIT setup (`ninja` inside the virtual environment) and forces Matplotlib's non-interactive `Agg` backend so Colab's inherited inline-backend setting cannot break the import check.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
VENV_PY = VENV / 'bin/python'
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([
    UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
    '--constraint', str(constraints), '-e', f'{REPO}[bagel]', 'matplotlib==3.10.7',
], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
ninja_path = VENV / 'bin/ninja'
assert ninja_path.exists(), f'Missing required FlashInfer JIT tool: {ninja_path}'
subprocess.run([str(ninja_path), '--version'], check=True)
for test_path in (
    'test/modular/test_bagel_checkpoint_revision.py',
    'test/modular/test_execution_trace.py',
    'test/modular/test_bagel_image_batch_packing.py',
    'test/modular/test_bagel_batch_parity.py',
):
    subprocess.run([str(VENV_PY), str(REPO / test_path), '-v'], cwd=REPO, check=True)
import_check = subprocess.check_output([
    str(VENV_PY), '-c',
    'import os; os.environ["MPLBACKEND"]="Agg"; import torch, flashinfer, matplotlib; assert torch.cuda.is_available(); x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, flashinfer.__version__, matplotlib.__version__, float(y[0,0]))'
], text=True)
(RESULT_DIR / 'gpu_import_check.txt').write_text(import_check)
freeze = subprocess.check_output([UV, 'pip', 'freeze', '--python', str(VENV_PY)], text=True)
(RESULT_DIR / 'resolved_requirements.txt').write_text(freeze)
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'installed_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '60',
], check=True)
print('Environment installation, CPU tests, and GPU import preflight passed')

## 4. Download and verify the pinned BAGEL checkpoint

In [ ]:
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
    '--cache-dir', str(CACHE_DIR),
    '--manifest', str(RESULT_DIR / 'checkpoint_manifest.json'),
], check=True)
import yaml
manifest = json.loads((RESULT_DIR / 'checkpoint_manifest.json').read_text())
pinned_config = yaml.safe_load((REPO / 'experiments/bagel/configs/bagel_single_gpu_pinned.yaml').read_text())
assert manifest['resolved_revision'] == pinned_config['model_kwargs']['checkpoint_revision']
print('Pinned checkpoint is available at', manifest['resolved_revision'])

## 5. Run the paired B=1 versus B=2 correctness gate

This cell starts and stops its own M* server for each arm. Expect the first server start to include model loading and FlashInfer compilation. Do not rerun this cell while it is still active.

In [ ]:
assert not PARITY_DIR.exists(), f'Result directory already exists; start a fresh RUN_ID instead of appending: {PARITY_DIR}'
import socket
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe_socket:
    try:
        probe_socket.bind(('127.0.0.1', 8000))
    except OSError as exc:
        raise RuntimeError('Port 8000 is busy. Stop the earlier M* process before this run.') from exc
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_batch_parity.py'),
    '--repo-root', str(REPO),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(PARITY_DIR),
    '--num-requests', '2',
    '--stagger-seconds', '8',
    '--batch-size', '2',
], cwd=REPO, check=True)
print('Paired heterogeneous-step parity run completed:', PARITY_DIR)

## 6. Inspect the gate and the paired images

The notebook reports exact hashes and pixel-error metrics without imposing an arbitrary numerical tolerance. We will interpret the measured differences before deciding whether the prototype passes numerical continuation.

In [ ]:
orchestrator = json.loads((PARITY_DIR / 'parity_orchestrator.json').read_text())
parity = json.loads((PARITY_DIR / 'serial_batch_parity.json').read_text())
assert orchestrator['status'] == 'success', orchestrator
assert parity['status'] == 'complete', parity
assert parity['batched_trace']['heterogeneous_step_image_completions'] > 0, parity['batched_trace']
print(json.dumps({
    'resolved_commit': head,
    'orchestrator_status': orchestrator['status'],
    'elapsed_seconds': round(orchestrator['elapsed_seconds'], 3),
    'comparison_summary': parity['summary'],
    'batched_trace': parity['batched_trace'],
    'result_dir': str(PARITY_DIR),
}, indent=2))

serial_summary = json.loads((PARITY_DIR / 'serial_b1/profile_summary.json').read_text())
batched_summary = json.loads((PARITY_DIR / 'heterogeneous_b2/profile_summary.json').read_text())
serial_by_index = {row['request_index']: row for row in serial_summary['measured_requests']}
batched_by_index = {row['request_index']: row for row in batched_summary['measured_requests']}
comparison_by_index = {row['request_index']: row for row in parity['requests']}
from IPython.display import Image, display
for request_index in sorted(comparison_by_index):
    comparison = comparison_by_index[request_index]
    print('\nRequest', request_index, json.dumps(comparison, indent=2))
    print('Serial B=1')
    display(Image(filename=serial_by_index[request_index]['image']['path'], width=384))
    print('Heterogeneous B=2')
    display(Image(filename=batched_by_index[request_index]['image']['path'], width=384))

## 7. Package the evidence for download

The original result directory remains in Drive. This cell also creates and downloads a ZIP so the complete evidence can be inspected locally before the runtime is deleted.

In [ ]:
archive_base = RESULT_DIR.parent / f'{RUN_ID}_results'
archive_path = shutil.make_archive(
    str(archive_base), 'zip', root_dir=RESULT_DIR.parent, base_dir=RESULT_DIR.name
)
print('Saved ZIP:', archive_path)
from google.colab import files
files.download(archive_path)

After the ZIP download finishes and the Drive result directory is visible, use **Runtime → Disconnect and delete runtime**. Share the downloaded ZIP. If the paired-run cell fails, download the result directory anyway: `parity_orchestrator.json`, the two server logs, and the execution traces contain the diagnostic evidence.